## 一、动态monkey patch
- 即在运行中给对象增加功能，区别于在 `class Student` 里把 `set_age` 写死，从一开始这个类就自带这个功能
1. **给某个具体实例绑定方法**

In [6]:
# 定义类
class Student(object):
    pass

# 实例
s = Student()
# 定义一个函数作为实例方法
def set_age(self, age):
    self.age = age

from types import MethodType  # 导入工具类
s.set_age = MethodType(set_age, s)   # `MethodType` 的作用：**把一个普通函数包装成绑定到某个具体实例的方法**
s.set_age(25)  # 调用时不用传 self
print(s.age )   # 只有这一个实例可用

# 尝试建一个新实例
s2 = Student()
s2.set_age(25)  # 尝试调用方法

25


AttributeError: 'Student' object has no attribute 'set_age'

2. **给class绑定方法**

In [7]:
# 定义函数
def set_score(self, score):
    self.score = score

# 绑定
Student.set_score = set_score

# 给class绑定方法后，所有实例均可调用：
s.set_score(100)
print(s.score)

s2.set_score(99)
print(s2.score)

100
99


### 使用`__slots__`
- 默认情况下，**每一个实例对象都会自带一个独立的字典 `__dict__`**，用来动态添加任意属性, `print(s.__dict__)   # {'name': 'Tom'}`.缺点：**字典占用内存很大**<br
                                                                                                                >
`__slots__` 就是用来解决这个内存浪费问题的。
- 在类里面定义 `__slots__ = (属性名列表)`,可以 ①取消实例的 `__dict__` ② 规定这个实例不能随便新增别的属性，节省内存

In [9]:
class Student:
    __slots__ = ('name', 'age') # 只允许 name、age 两个属性

s = Student()
s.name = "Alice"
s.age = 18

s.score = 90   # 报错！__slots__没有score，不能新增属性

AttributeError: 'Student' object has no attribute 'score'

==使用__slots__要注意，__slots__定义的属性仅对当前类实例起作用，**对继承的子类是不起作用的**：==

In [11]:
class GraduateStudent(Student):
    pass

g = GraduateStudent()
g.score = 9999    # 除非在子类也加上__slots__

## 二、类的装饰器`@property`
**使用场景**
1. **只读属性**：只写 @property，不写 setter，外部不能修改
2. **数据校验**：赋值时过滤非法值
3. **动态计算属性**：值不存放在变量里，每次实时计算

In [6]:
# 场景一：读取,不可修改 -- 只有property
class Person:
    def __init__(self, age):
        self._age = age

    @property
    def age(self):        # 加了 `@property` 的方法**不能带参数（除 self）**
        return self._age

p = Person(19)
print(p.age)    # 调用时不用 (),从代码来看是方法，但使用时与属性一样

19


In [7]:
# 场景二：写 -- 给属性赋值（修改 + 校验）property + setter
class Person:
    def __init__(self, age):
        self._age = age

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        # 赋值时做合法性校验
        if not isinstance(value, int):
            raise ValueError("年龄必须是整数")
        if value < 0 or value > 150:
            raise ValueError("年龄范围不合理")
        self._age = value


p2 = Person(10)
p2.age = 20     # 修改
print(p2.age)

p2.age = 200  # 抛出异常，实现数据校验

20


ValueError: 年龄范围不合理

In [8]:
# 场景三：动态计算属性：值不存放在变量里，每次实时计算
class Rectangle:
    def __init__(self, w, h):
        self.width = w
        self.height = h

    @property
    def area(self):
        # 动态计算，不需要存一个area变量
        return self.width * self.height

r = Rectangle(3,4)
print(r.area)
r.width = 5
print(r.area)


12
20


==注意：如果 `@property` 装饰的方法名 = 你返回 / 赋值用的实例变量名，**会无限递归，爆栈 RecursionError**。==

In [9]:
# 错误写法
class Student(object):
    @property
    def birth(self):
        return self.birth    # 变量应写成_birth
# setter也同样不能重名

In [12]:
# 练习
# 请利用@property给一个Screen对象加上width和height属性，以及一个只读属性resolution：
class Screen(object):
    def __init__(self):
        self._width = 0
        self._height = 0

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if not isinstance(value, (int, float)):
            raise ValueError("width must be a number")
        self._width = value
    

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        if not isinstance(value, (int, float)):
            raise ValueError("height must be a number")
        self._height = value

    @property
    def resolution(self):
        return self._width * self._height  # 动态计算

# 测试:
s = Screen()
s.width = 1024
s.height = 768
print('resolution =', s.resolution)
if s.resolution == 786432:
    print('测试通过!')
else:
    print('测试失败!')


resolution = 786432
测试通过!


## 三、多重继承MixIn
- 一个类可以继承多个父类的功能

In [1]:
# 首先是Animal类，分为Mammal和Bird大类
class Animal(object):
    pass

# 大类:
class Mammal(Animal):
    pass

class Bird(Animal):
    pass

# 各种动物:
class Dog(Mammal):
    pass

class Bat(Mammal):
    pass

# 要给动物再加上Runnable和Flyable的功能，只需要先定义好Runnable和Flyable的类：
class Runnable(object):
    def run(self):
        print('Running...')

class Flyable(object):
    def fly(self):
        print('Flying...')

# 对于需要Runnable功能的动物，就多继承一个Runnable
class Dog(Mammal, Runnable):
    pass

class Bat(Mammal, Flyable):
    pass

### Mixin功能插件
- `MixIn`是一种特殊的父类，**只提供方法、不维护实例属性**，用来给多个不同类 “附加功能”，而不用继承一个庞大的公共父类。
- 类名一般以`MixIn / Mixin`结尾，用来标识它**只是功能插件，不能单独实例化**。

In [3]:
# Mixin：只提供打印日志的功能
class LogMixin:
    def log(self, msg):
        print(f"[{self.__class__.__name__}] {msg}")  # self.__class__获取所属的类，加__name__获取类名

# 普通业务类，混入日志能力
class User(LogMixin):    # 赋予User功能
    def __init__(self, name):
        self.name = name

u = User("Tom")
u.log("创建用户")   # 直接拿到 log 方法


[User] 创建用户


In [ ]:
# 实际应用
from socketserver import TCPServer, UDPServer, ForkingMixIn, ThreadingMixIn

# 多进程TCP服务器
class MyTCPServer(TCPServer, ForkingMixIn):
    pass

# 多线程UDP服务器
class MyUDPServer(UDPServer, ThreadingMixIn):
    pass


#### MRO顺序
如果多个父类有同名方法，**写在左边的类优先级更高**。
规范写法：**Mixin 写在前面**，`class MyTCPServer(ForkingMixIn, TCPServer):`,保证 Mixin 的并发逻辑优先执行

## 四、定制类
|魔术方法|作用|触发时机|代码示例|
|:----------:|:----------:|:---------:|:-------:|
| `__repr__(self)` | 返回**面向开发者**的精准对象表示 | 交互式控制台直接输出对象、`repr(obj)` |
| `__iter__(self)` | 将实例变为**可迭代对象**，提供迭代器 | 执行 `for x in obj`、`iter(obj)` |
| `__getitem__(self, key)` | 支持下标/切片取值 `obj[key]` | 使用`obj[索引/键/切片]`访问元素 |
| `__getattr__(self, name)` | 捕获**不存在属性**的访问请求 | 访问`obj.xxx`，xxx在对象中不存在 |
| `__call__(self, *args, **kwargs)` | 让实例**可以像函数一样加括号调用** | `obj(参数)` |

[详情](https://liaoxuefeng.com/books/python/oop-adv/special-method/index.html)

In [5]:
# __call__示例 --- 深度学习常用
class Demo:
    def __call__(self, x):  
        return x * 2

d = Demo()
res = d(10)   # 调用 __call__，实例变成可调用对象callable
print(res)    # 20

20


## 五、枚举类Enum
- 枚举：固定、有限、不可修改的常量集合。比如：星期、性别、订单状态<br>
需要导入：`from enum import Enum`

In [7]:
# 基础写法
from enum import Enum, unique

@unique      # 可以帮助我们检查保证没有重复值 , 但其实没有硬性要求加unique
class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

print(Color.RED)        # Color.RED
print(Color.RED.name)   # RED  名字（字符串）
print(Color.RED.value)  # 1    保存的值


Color.RED
RED
1


### 获取枚举的方式
1. **通过类名点访问**

In [8]:
c = Color.RED

2. **通过值反向查找 `Enum(值)`**

In [10]:
print(Color(1))   # Color.RED

Color.RED


3. **通过名字字符串查找 `Enum["名字"]`**

In [9]:
print(Color["RED"]) # Color.RED

Color.RED


In [11]:
# 遍历枚举
for item in Color:
    print(item.name, item.value)

RED 1
GREEN 2
BLUE 3


## 六、使用元类